# Solutions: A minimal BERTopic pipeline

Reference solutions for `01_minimal-pipeline.ipynb`. Try the exercises yourself before reading these.

In [ ]:
# Colab only
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/clariah2025-dse-ml/materials/TODO-path/

### Exercise 1

In [ ]:
!pip install bertopic

import pandas as pd
from bertopic import BERTopic

df = pd.read_csv('dh2005-2026.csv')

documents = df[(df.abstract.notnull()) & (df.language == 'en')].abstract.to_list()

print(len(documents), "documents")

The filter matters: some rows have no abstract at all, and the corpus includes other languages besides English. Feeding those into an English-language model would just add noise.

### Exercise 2

In [ ]:
topic_model = BERTopic(verbose=True, language="english")
topics, probs = topic_model.fit_transform(documents)

`topics` holds one topic id per document (same order as `documents`); `probs` holds how confident the model is about each assignment.

### Exercise 3

In [ ]:
topic_info = topic_model.get_topic_info()

largest_topic_id = int(
    topic_info[topic_info.Topic != -1]
    .sort_values("Count", ascending=False)
    .iloc[0]
    .Topic
)

topic_model.get_topic(largest_topic_id)

Topic `-1` is not a real topic: it's BERTopic's bucket for documents the clustering step couldn't confidently assign anywhere. It's usually the biggest group by count, so it has to be excluded before picking the largest *actual* topic, or you'd just get the outliers back.

The `int(...)` around the whole expression matters more than it looks: `.iloc[0].Topic` returns a `numpy.int64`, not a plain Python `int`. `get_topic()` doesn't care about that difference, but `get_representative_docs()` in Exercise 6 does a strict `isinstance(topic, int)` check internally, which a `numpy.int64` fails. Without this cast, Exercise 6 doesn't error, it silently returns something else entirely (see the note there).

### Exercise 4

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(width=280, height=330, top_n_topics=15, n_words=10)

### Exercise 5

In [ ]:
topic_model.save("dh-dataset.mm")

reloaded_model = BERTopic.load("dh-dataset.mm")

reloaded_model.get_topic_info().equals(topic_model.get_topic_info())

`.equals()` checks the two tables are actually identical, not just similar-looking when printed. It should return `True`.

### Exercise 6

In [ ]:
representative_docs = topic_model.get_representative_docs(largest_topic_id)

for doc in representative_docs:
    print(doc, "\n")

`get_representative_docs(topic_id)` returns (by default, up to 3) of the actual input documents BERTopic found most representative of that topic, pulled from the `representative_docs_` it computed during `fit_transform`.

If this cell instead prints a short list of plain numbers (`-1`, `0`, `1`, ...) rather than abstract text, `largest_topic_id` was passed in as a `numpy.int64` instead of a plain `int` (see the note in Exercise 3). BERTopic's source does `if isinstance(topic, int): ... else: return self.representative_docs_`, so a `numpy.int64` fails that check silently and the method hands back its *entire* dictionary of all topics instead of raising an error. A `for` loop over a dict iterates its keys, which is exactly the sequence of topic ids you'd see. Called with no argument at all, it returns that same dictionary, covering every topic at once, which is what's actually happening here.

### Exercise 7

In [ ]:
for doc in representative_docs:
    matches = df[df.abstract == doc]
    print(matches[["year", "title"]])
    print()

`documents` was built straight from `df.abstract`, with nothing altered in between, so an exact string match (`df.abstract == doc`) finds the source row. `matches` is a DataFrame rather than a single row on purpose: if the same abstract text happens to appear more than once in `df` (a reprint, a duplicate entry), `==` returns every matching row rather than silently picking one, worth checking for before assuming there's exactly one.